# A 同学评估 notebook
数据几何结构与模型容量对去噪模型记忆和泛化的影响

**流程**：0 环境 → 1 配置 → 2 自检 → 3 验收 npz → 4 pilot 分析 → 5 主实验指标 → 6 质检 → 7 轨迹图 → 8 推送

指标只算三项（mode_coverage / mmd / nn_ratio），按 B 的约定另存 `metrics.csv`，**不改 B 的 results_B.csv**。

## 0. 环境：挂载 Drive、拉取分支 feature/A

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from getpass import getpass

GITHUB_USER = "MC648472"
REPO        = "CISC7402-denoise-memorization"
BRANCH      = "feature/A"
ROOT        = f"/content/{REPO}"

if not os.path.exists(ROOT):
    token = getpass("GitHub token（不显示，也别写进代码）: ")
    !git clone https://{token}@github.com/{GITHUB_USER}/{REPO}.git {ROOT}
%cd {ROOT}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}

sys.path.insert(0, f"{ROOT}/code")

## 1. 配置
`DATA_ROOT` 是 B 共享出来的 `diffusion_project/`（里面有 pilot、pilot2、main 和 results_B.csv）。

In [ ]:
import importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import eval_toolkit as ev
importlib.reload(ev)

DATA_ROOT = "/content/drive/MyDrive/diffusion_project"
METRICS   = f"{DATA_ROOT}/metrics.csv"        # A 算完另存，不碰 results_B.csv

DIST_COL, CAP_COL, SEED_COL, PATH_COL = "radius", "width", "seed", "npz_path"
A_DIST_IDX = [0, 1, 2]                       # A 跑距离下标 0,1,2；B 跑 3,4
N_CAP, N_SEED = 5, 5
EXPECT_A = len(A_DIST_IDX) * N_CAP * N_SEED  # 75

print("K_MODES =", ev.K_MODES, "| SIGMA 兜底 =", ev.SIGMA)
print("注意：CSV 里的 radius 是环半径，中心间距 = 0.7654*radius")
print("A 应完成:", EXPECT_A, "次")

## 2. 自检（假数据）
确认三项指标的已知行为：perfect≈(1.0, 0.000, 0.05)、memorize 的 nn_ratio≈1、collapse3 的 coverage=0.375。

In [ ]:
ev.selftest()

## 3. 验收 B 的 npz
B 的文件里只有 `final / traj / train / losses / means_radius`，缺 `dist`、`seed` 只是 WARN（靠 CSV 对号），数组或轨迹不对才是 FAIL。

In [ ]:
!python {ROOT}/code/check_npz.py {DATA_ROOT}/pilot2 --quiet

## 4. Pilot 分析（tag=pilot2）
先算指标写进 metrics.csv，再出距离×容量两张表，找崩塌区间和饱和容量点。

In [ ]:
ev.build_metrics([f"{DATA_ROOT}/results_B.csv"], out=METRICS,
                 npz_root=DATA_ROOT, tags=["pilot2"])

m = pd.read_csv(METRICS)
pilot = m[m.tag == "pilot2"].merge(
    pd.read_csv(f"{DATA_ROOT}/results_B.csv")[["tag", "dist_idx", "cap_idx", "seed",
                                              "radius", "width", "depth"]],
    on=["tag", "dist_idx", "cap_idx", "seed"], how="left")
ev.pilot_report(pilot, DIST_COL, CAP_COL)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, met in zip(axes, ["mode_coverage", "mmd", "nn_ratio"]):
    t = pilot.pivot_table(index=DIST_COL, columns="cap_idx", values=met)
    im = ax.imshow(t.values, aspect="auto", origin="lower")
    ax.set_xticks(range(len(t.columns))); ax.set_xticklabels(t.columns)
    ax.set_yticks(range(len(t.index)));   ax.set_yticklabels(t.index)
    ax.set_xlabel("cap_idx"); ax.set_ylabel("radius"); ax.set_title(met)
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 5. 主实验：算指标（可反复运行，只补没算过的）
A 的 npz 在 `main/`，B 的在同目录；两份 CSV 一起读，靠 `tag/dist_idx/cap_idx/seed` 对齐。

In [ ]:
CSVS = [f"{DATA_ROOT}/results_A.csv", f"{DATA_ROOT}/results_B.csv"]
CSVS = [c for c in CSVS if os.path.exists(c)]
ev.build_metrics(CSVS, out=METRICS, npz_root=DATA_ROOT, tags=["main"])

## 6. 质检：缺格、越界、种子间波动、训练是否收敛

In [ ]:
m = pd.read_csv(METRICS)
main = m[m.tag == "main"]
print(f"指标行数 {len(main)} / 125")
print(main.groupby(["dist_idx", "cap_idx"]).size().unstack("cap_idx").to_string())

bad = main[(main.mode_coverage < 0) | (main.mode_coverage > 1) |
           (main.nn_ratio < 0) | (main.nn_ratio > 1)]
print("\n越界行数:", len(bad))
sd = main.groupby(["dist_idx", "cap_idx"])[["mode_coverage", "mmd", "nn_ratio"]].std()
print("\n种子间标准差最大的 5 个格子:")
print(sd.sort_values("mmd", ascending=False).head().round(4).to_string())

In [ ]:
res = pd.concat([pd.read_csv(c) for c in CSVS])
res = res[res.tag == "main"]
print("final_loss 范围:", res.final_loss.min(), "~", res.final_loss.max())
print("单次耗时(s):", res.seconds.describe()[['min','50%','max']].to_dict())

## 7. 轨迹追踪 & 样本散点图
选一个 run：①训练集 vs 生成样本 ②到训练集的近邻距离 ③近邻比 ④模式命中率。

In [ ]:
row = res.iloc[0]                                  # 改成想看的那一行
z = np.load(os.path.join(DATA_ROOT, os.path.basename(row[PATH_COL])))
r, s = float(row[DIST_COL]), int(row[SEED_COL])
centers = ev.gmm_centers(ev.chord(r))
sigma = float(z["sigma"]) if "sigma" in z.files else ev.estimate_sigma(z["train"], centers)

ref = ev.reference_sets(ev.chord(r), sigma, s)
tau = ev.nn_threshold(ref["calib"], z["train"])
tm = ev.traj_metrics(z["traj"], z["train"], tau, centers, sigma)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].scatter(*z["train"].T, s=3, alpha=.4, label="train")
ax[0].scatter(*z["final"].T, s=3, alpha=.4, label="generated")
ax[0].scatter(*centers.T, marker="x", c="k"); ax[0].legend(); ax[0].set_title("samples")
for a, k in zip(ax[1:], ["mean_nn_dist", "nn_ratio", "mode_hit"]):
    a.plot(tm[k], marker="o"); a.set_title(k); a.set_xlabel("snapshot (0=noise)")
plt.tight_layout(); plt.show()

## 8. 推送（只推代码和 csv；npz 太big，`.gitignore` 里应有 `*.npz`）

In [ ]:
!git add -A
!git commit -m "A: eval update" || echo "nothing to commit"
!git push origin {BRANCH}